# Notebook 2: Text Preprocessing

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the second notebook of the project.

In Notebook 1, we downloaded and standardized the BBC News dataset.

In this notebook, we will clean the article text before document chunking.

The goal of this notebook is to:

1. Load the standardized dataset from Notebook 1.
2. Clean the article text.
3. Remove unnecessary spaces, HTML tags, URLs, and special formatting.
4. Keep the text natural for embedding models.
5. Compare original and cleaned text.
6. Save the preprocessed dataset for the next notebook.

The output file from this notebook will be:

`bbc_docs_preprocessed.csv`

This file will be used in:

`3 - Document Chunking.ipynb`

In [25]:
# Import required libraries
import os
import re
import pandas as pd



# Load dataset
def load_dataset(file_path):
    """
    Loads the standardized dataset created in Notebook 1.
    
    Parameters:
        file_path (str): Path to input CSV file
        
    Returns:
        DataFrame: Loaded dataset
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"{file_path} not found. Please run Notebook 1 first."
        )
    
    df = pd.read_csv(file_path)
    
    print("Dataset loaded successfully.")
    print("Input file:", file_path)
    print("Dataset shape:", df.shape)
    
    return df


# Clean article text
def clean_text(text):
    """
    Applies light text cleaning while preserving meaning.
    
    This function removes:
    - HTML tags
    - URLs
    - Extra spaces
    - Extra line breaks
    - Repeated punctuation spacing
    
    It does NOT remove stopwords, stem words, or lemmatize text.
    """
    text = str(text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # Replace new lines and tabs with spaces
    text = re.sub(r"[\n\r\t]+", " ", text)
    
    # Remove extra spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    
    # Add space after punctuation if missing
    text = re.sub(r"([.,!?;:])([A-Za-z])", r"\1 \2", text)
    
    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading and trailing spaces
    text = text.strip()
    
    return text


# Preprocess full dataset
def preprocess_documents(df, min_words=20):
    """
    Cleans document text and removes very short documents.
    
    Parameters:
        df (DataFrame): Input document dataframe
        min_words (int): Minimum number of words required
        
    Returns:
        DataFrame: Preprocessed dataframe
    """
    df = df.copy()
    
    # Clean title and text
    df["title"] = df["title"].fillna("Untitled Article").astype(str).apply(clean_text)
    df["text_clean"] = df["text"].fillna("").astype(str).apply(clean_text)
    
    # Add word count after cleaning
    df["clean_word_count"] = df["text_clean"].apply(lambda x: len(x.split()))
    
    # Remove very short documents
    df = df[df["clean_word_count"] >= min_words].reset_index(drop=True)
    
    # Recreate document IDs after filtering
    df["doc_id"] = range(1, len(df) + 1)
    
    return df


# Show text comparison
def show_text_comparison(df, index=0):
    """
    Displays original and cleaned text for comparison.
    """
    row = df.iloc[index]
    
    print("Document ID:", row["doc_id"])
    print("Title:", row["title"])
    print("\nOriginal Text Preview:\n")
    print(row["text"][:700])
    
    print("\nCleaned Text Preview:\n")
    print(row["text_clean"][:700])


# Save dataset
def save_dataset(df, output_path):
    """
    Saves the preprocessed dataset as CSV.
    """
    df.to_csv(output_path, index=False)
    
    print("Dataset saved successfully.")
    print("Output file:", output_path)




In [27]:
# Load standardized dataset from Notebook 1
input_file = "/kaggle/input/datasets/jahnavidulala/bbc-docs-standard/bbc_docs_standard.csv"
df_docs = load_dataset(input_file)


# Preview input dataset
print("\nInput Dataset Preview:")
display(df_docs.head())

print("\nInput Dataset Info:")
print("Shape:", df_docs.shape)
print("Columns:", df_docs.columns.tolist())
print("\nMissing Values:")
print(df_docs.isnull().sum())


# Apply text preprocessing
df_preprocessed = preprocess_documents(df_docs, min_words=20)


# Preview cleaned dataset
print("\nPreprocessed Dataset Preview:")
display(df_preprocessed.head())

print("\nPreprocessed Dataset Shape:", df_preprocessed.shape)


# Text length summary

print("\nCleaned Word Count Summary:")
print(df_preprocessed["clean_word_count"].describe())


# Compare original and cleaned text
print("\nOriginal vs Cleaned Text Example:")
show_text_comparison(df_preprocessed, index=0)


# Keep final useful columns
final_columns = [
    "doc_id",
    "title",
    "category",
    "text",
    "text_clean",
    "word_count",
    "clean_word_count"
]

# Keep only columns that exist
final_columns = [col for col in final_columns if col in df_preprocessed.columns]

df_preprocessed = df_preprocessed[final_columns]


# Save preprocessed dataset
output_file = "bbc_docs_preprocessed.csv"

save_dataset(df_preprocessed, output_file)



# Verify saved file
saved_df = pd.read_csv(output_file)

print("\nSaved file verified successfully.")
print("Saved file shape:", saved_df.shape)

display(saved_df.head())

Dataset loaded successfully.
Input file: /kaggle/input/datasets/jahnavidulala/bbc-docs-standard/bbc_docs_standard.csv
Dataset shape: (38730, 5)

Input Dataset Preview:


,doc_id,title,category,text,word_count
0,1,Ukraine: Angry Zelensky vows to punish Russian...,unknown,The Ukrainian president says the country will ...,16
1,2,War in Ukraine: Taking cover in a town under a...,unknown,"Jeremy Bowen was on the frontline in Irpin, as...",18
2,3,Ukraine war 'catastrophic for global food',unknown,One of the world's biggest fertiliser firms sa...,17
3,4,Manchester Arena bombing: Saffie Roussos's par...,unknown,The parents of the Manchester Arena bombing's ...,16
4,5,Ukraine conflict: Oil price soars to highest l...,unknown,Consumers are feeling the impact of higher ene...,16



Input Dataset Info:
Shape: (38730, 5)
Columns: ['doc_id', 'title', 'category', 'text', 'word_count']

Missing Values:
doc_id        0
title         0
category      0
text          0
word_count    0
dtype: int64

Preprocessed Dataset Preview:


,doc_id,title,category,text,word_count,text_clean,clean_word_count
0,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21,More than 1.5 million Ukrainians have fled the...,21
1,2,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31,Russian gymnast Ivan Kuliak is being investiga...,31
2,3,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21,Several US presidents have failed to get the m...,21
3,4,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20,A ceasefire agreement in the southern city of ...,20
4,5,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20,The moment a man swims out of the path of a co...,20



Preprocessed Dataset Shape: (8622, 7)

Cleaned Word Count Summary:
count    8622.000000
mean       23.374971
std         3.356254
min        20.000000
25%        21.000000
50%        22.000000
75%        25.000000
max        45.000000
Name: clean_word_count, dtype: float64

Original vs Cleaned Text Example:
Document ID: 1
Title: Ukraine conflict: Your guide to understanding day 11

Original Text Preview:

More than 1.5 million Ukrainians have fled the country. Here's what you need to know after day 11 of the war.

Cleaned Text Preview:

More than 1.5 million Ukrainians have fled the country. Here's what you need to know after day 11 of the war.
Dataset saved successfully.
Output file: bbc_docs_preprocessed.csv

Saved file verified successfully.
Saved file shape: (8622, 7)


,doc_id,title,category,text,text_clean,word_count,clean_word_count
0,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,More than 1.5 million Ukrainians have fled the...,21,21
1,2,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,Russian gymnast Ivan Kuliak is being investiga...,31,31
2,3,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,Several US presidents have failed to get the m...,21,21
3,4,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,A ceasefire agreement in the southern city of ...,20,20
4,5,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,The moment a man swims out of the path of a co...,20,20
